## 地下水深度、地下水涵養量データのダウンロード

In [2]:
# 必要なら最初に別セルで実行
# %pip install -U xarray netCDF4 h5netcdf

from __future__ import annotations

import os
import re
import shutil
import xml.etree.ElementTree as ET
from io import StringIO
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote, unquote, urlparse
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# 0. 基本設定
# ============================================================

# サーバー上での保存先
OUTPUT_DIR = Path(
    os.environ.get(
        "GROUNDWATER_OUTPUT_DIR",
        "/work/tsuda/GAEZ/WaterData/groundwater",
    )
)

# Windowsで実行する場合の例
if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = Path(
        r"C:\masterresearch\Comparative_advantage\GAEZ"
    ) / "WaterData" / "groundwater"

WTD_RAW_DIR = OUTPUT_DIR / "fan_wtd" / "raw_continents"
WTD_SUBSET_DIR = OUTPUT_DIR / "fan_wtd" / "regional_subsets"

RECHARGE_RAW_DIR = OUTPUT_DIR / "watergap_2_2d" / "raw"
RECHARGE_PROCESSED_DIR = OUTPUT_DIR / "watergap_2_2d" / "processed"

for directory in [
    OUTPUT_DIR,
    WTD_RAW_DIR,
    WTD_SUBSET_DIR,
    RECHARGE_RAW_DIR,
    RECHARGE_PROCESSED_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# 地域診断用の範囲
REGION_BOXES = {
    "us_west_100w": (-110.0, -90.0, 25.0, 50.0),
    "nile": (-10.0, 40.0, 5.0, 32.0),
    "yellow_river": (100.0, 125.0, 25.0, 45.0),
    "japan": (122.0, 154.0, 20.0, 47.0),
}

REGION_CONTINENT = {
    "us_west_100w": "north_america",
    "nile": "africa",
    "yellow_river": "asia",
    "japan": "asia",
}

# まずは地域診断に必要な大陸だけ取得
# 全球モデル用に全大陸を取得する場合は "all" に変更
WTD_DOWNLOAD_MODE = "diagnostic_regions"

# WaterGAPの平均期間
# 2020年のデータではなく、利用可能な期間内の平均です。
RECHARGE_MEAN_START_YEAR = 2000
RECHARGE_MEAN_END_YEAR = 2010

print("保存先:", OUTPUT_DIR)
print("WTD mode:", WTD_DOWNLOAD_MODE)


# ============================================================
# 1. 共通ダウンロード関数
# ============================================================

def download_file(
    url: str,
    destination: Path,
    *,
    headers: dict[str, str] | None = None,
    timeout: int = 1200,
) -> Path:
    """
    完了済みファイルがあれば再利用する。
    途中ファイルを残さないよう、.part経由で保存する。
    """
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0:
        print("再利用:", destination)
        return destination

    temporary_path = destination.with_suffix(
        destination.suffix + ".part"
    )

    request_headers = {
        "User-Agent": "Mozilla/5.0 groundwater-download",
    }

    if headers:
        request_headers.update(headers)

    print("ダウンロード中:", url)

    request = Request(url, headers=request_headers)

    try:
        with urlopen(request, timeout=timeout) as response:
            with temporary_path.open("wb") as output_file:
                shutil.copyfileobj(response, output_file)

        if not temporary_path.exists() or temporary_path.stat().st_size == 0:
            raise RuntimeError(f"空のファイルです: {url}")

        temporary_path.replace(destination)

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    print(
        "保存:",
        destination,
        "size=",
        destination.stat().st_size,
        "bytes",
    )

    return destination


def read_url_bytes(url: str, timeout: int = 180) -> bytes:
    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 groundwater-download",
        },
    )

    with urlopen(request, timeout=timeout) as response:
        return response.read()


# ============================================================
# 2. Fan地下水面深度（WTD）の取得
# ============================================================

WTD_CATALOG_URLS = [
    (
        "https://thredds-gfnl.usc.es/thredds/catalog/"
        "GLOBALWTDFTP/annualmeans/catalog.xml"
    ),
    (
        "http://thredds-gfnl.usc.es/thredds/catalog/"
        "GLOBALWTDFTP/annualmeans/catalog.xml"
    ),
]

WTD_FILE_SERVER_BASES = [
    "https://thredds-gfnl.usc.es/thredds/fileServer/",
    "http://thredds-gfnl.usc.es/thredds/fileServer/",
]


def xml_local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1]


def get_wtd_catalog():
    last_error = None

    for catalog_url in WTD_CATALOG_URLS:
        try:
            xml_bytes = read_url_bytes(catalog_url)
            return ET.fromstring(xml_bytes), catalog_url
        except (
            HTTPError,
            URLError,
            TimeoutError,
            ET.ParseError,
        ) as error:
            print("カタログ取得失敗:", catalog_url)
            print(error)
            last_error = error

    raise RuntimeError(
        "Fan WTDのTHREDDSカタログを取得できませんでした。"
    ) from last_error


wtd_catalog_root, wtd_catalog_url_used = get_wtd_catalog()

wtd_dataset_paths = []

for element in wtd_catalog_root.iter():
    if xml_local_name(element.tag) != "dataset":
        continue

    url_path = element.attrib.get("urlPath")

    if url_path and url_path.lower().endswith(".nc"):
        wtd_dataset_paths.append(url_path)

wtd_dataset_paths = sorted(set(wtd_dataset_paths))

if not wtd_dataset_paths:
    raise RuntimeError(
        "WTDカタログからNetCDFファイルが見つかりません。"
    )

print("WTD catalog:", wtd_catalog_url_used)
print("WTD files:")

for path in wtd_dataset_paths:
    print(" -", path)


CONTINENT_PATTERNS = {
    "north_america": [
        "north_america",
        "northamerica",
        "namerica",
    ],
    "south_america": [
        "south_america",
        "southamerica",
        "samerica",
    ],
    "africa": ["africa"],
    "asia": ["asia"],
    "europe": ["europe"],
    "australia": [
        "australia",
        "oceania",
    ],
}


def choose_wtd_dataset(continent: str) -> str:
    patterns = CONTINENT_PATTERNS[continent]
    candidates = []

    for dataset_path in wtd_dataset_paths:
        filename = Path(dataset_path).name.lower()
        normalized_filename = re.sub(
            r"[^a-z0-9]+",
            "_",
            filename,
        )

        if any(pattern in normalized_filename for pattern in patterns):
            candidates.append(dataset_path)

    if not candidates:
        raise RuntimeError(
            f"{continent}のWTDファイルが見つかりません。\n"
            f"候補一覧: {wtd_dataset_paths}"
        )

    return sorted(candidates)[0]


if WTD_DOWNLOAD_MODE == "all":
    continents_to_download = list(CONTINENT_PATTERNS)

elif WTD_DOWNLOAD_MODE == "diagnostic_regions":
    continents_to_download = sorted(
        set(REGION_CONTINENT.values())
    )

else:
    raise ValueError(
        "WTD_DOWNLOAD_MODEは "
        "'diagnostic_regions' または 'all' にしてください。"
    )


wtd_raw_paths = {}

for continent in continents_to_download:
    dataset_path = choose_wtd_dataset(continent)
    destination = WTD_RAW_DIR / Path(dataset_path).name

    downloaded = False
    last_error = None

    for base_url in WTD_FILE_SERVER_BASES:
        try:
            download_file(
                base_url + dataset_path,
                destination,
                timeout=1200,
            )
            downloaded = True
            break
        except (
            HTTPError,
            URLError,
            TimeoutError,
        ) as error:
            print("WTDファイル取得失敗:", base_url)
            print(error)
            last_error = error

    if not downloaded:
        raise RuntimeError(
            f"{continent}のWTDファイル取得に失敗しました。"
        ) from last_error

    wtd_raw_paths[continent] = destination


wtd_manifest = pd.DataFrame(
    [
        {
            "continent": continent,
            "dataset_path": choose_wtd_dataset(continent),
            "local_path": str(path),
            "size_bytes": path.stat().st_size,
            "source": "Fan global water table depth",
            "definition": "depth from land surface to modeled water table",
            "temporal_status": "static/model estimate",
        }
        for continent, path in wtd_raw_paths.items()
    ]
)

wtd_manifest.to_csv(
    OUTPUT_DIR / "fan_wtd_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nWTD manifest")
print(wtd_manifest.to_string(index=False))


# ============================================================
# 3. WTDの地域別切り出し
# ============================================================

def find_coordinate_name(ds, candidates):
    exact = {
        name.lower(): name
        for name in ds.coords
    }

    for candidate in candidates:
        if candidate.lower() in exact:
            return exact[candidate.lower()]

    for name in ds.coords:
        if any(
            candidate.lower() in name.lower()
            for candidate in candidates
        ):
            return name

    raise KeyError(
        f"座標が見つかりません: {candidates}; "
        f"available={list(ds.coords)}"
    )


def find_wtd_variable(ds, lat_name, lon_name):
    candidates = []

    for name, variable in ds.data_vars.items():
        if lat_name in variable.dims and lon_name in variable.dims:
            candidates.append(name)

    if not candidates:
        raise KeyError(
            "緯度・経度を含むWTD変数がありません。"
            f"data_vars={list(ds.data_vars)}"
        )

    preferred = [
        name
        for name in candidates
        if any(
            token in name.lower()
            for token in ["wtd", "water", "depth"]
        )
    ]

    return preferred[0] if preferred else candidates[0]


def normalize_longitude(ds, lon_name):
    longitude = np.asarray(ds[lon_name].values)

    if np.nanmax(longitude) > 180.0:
        normalized = ((longitude + 180.0) % 360.0) - 180.0
        ds = ds.assign_coords(
            {lon_name: normalized}
        ).sortby(lon_name)

    return ds


def make_slice(coordinate, lower, upper):
    values = np.asarray(coordinate.values)

    if values[0] <= values[-1]:
        return slice(lower, upper)

    return slice(upper, lower)


def convert_wtd_to_meters(data):
    units = str(
        data.attrs.get("units", "")
    ).lower()

    values = data.astype("float32")

    if "cm" in units:
        values = values / 100.0

    elif "mm" in units:
        values = values / 1000.0

    values.attrs["original_units"] = units
    values.attrs["units"] = "m"
    values.attrs["definition"] = (
        "depth from land surface to modeled water table"
    )

    return values


def extract_wtd_region(
    raw_path: Path,
    region_name: str,
    bbox,
):
    west, east, south, north = bbox

    with xr.open_dataset(
        raw_path,
        decode_cf=True,
    ) as source_ds:

        lon_name = find_coordinate_name(
            source_ds,
            ["lon", "longitude", "x"],
        )

        ds = normalize_longitude(
            source_ds,
            lon_name,
        )

        lat_name = find_coordinate_name(
            ds,
            ["lat", "latitude", "y"],
        )

        lon_name = find_coordinate_name(
            ds,
            ["lon", "longitude", "x"],
        )

        variable_name = find_wtd_variable(
            ds,
            lat_name,
            lon_name,
        )

        data = ds[variable_name].sel(
            {
                lat_name: make_slice(
                    ds[lat_name],
                    south,
                    north,
                ),
                lon_name: make_slice(
                    ds[lon_name],
                    west,
                    east,
                ),
            }
        )

        non_spatial_dims = [
            dim
            for dim in data.dims
            if dim not in {lat_name, lon_name}
        ]

        if non_spatial_dims:
            data = data.mean(
                dim=non_spatial_dims,
                skipna=True,
            )

        data = (
            convert_wtd_to_meters(data)
            .rename("water_table_depth_m")
            .load()
        )

        data.attrs["source_variable"] = variable_name
        data.attrs["region"] = region_name

        output_path = (
            WTD_SUBSET_DIR
            / f"fan_wtd_{region_name}.nc"
        )

        data.to_dataset().to_netcdf(output_path)

    return output_path, data


wtd_region_summary_rows = []

for region_name, bbox in REGION_BOXES.items():
    continent = REGION_CONTINENT[region_name]
    raw_path = wtd_raw_paths.get(continent)

    if raw_path is None:
        print("WTD未取得のためスキップ:", region_name)
        continue

    output_path, data = extract_wtd_region(
        raw_path,
        region_name,
        bbox,
    )

    finite = data.values[np.isfinite(data.values)]

    wtd_region_summary_rows.append(
        {
            "region": region_name,
            "continent": continent,
            "path": str(output_path),
            "n_finite": finite.size,
            "min_m": float(np.min(finite)),
            "median_m": float(np.median(finite)),
            "p90_m": float(np.percentile(finite, 90)),
            "max_m": float(np.max(finite)),
        }
    )

wtd_region_summary = pd.DataFrame(
    wtd_region_summary_rows
)

wtd_region_summary.to_csv(
    OUTPUT_DIR / "fan_wtd_regional_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nWTD regional summary")
print(wtd_region_summary.to_string(index=False))


# ============================================================
# 4. WaterGAP 2.2d涵養量の候補ファイルを取得
# ============================================================

PANGAEA_DOI = "10.1594/PANGAEA.918447"
PANGAEA_DATASET_ID = "918447"

PANGAEA_TABLE_URL = (
    f"https://doi.pangaea.de/{PANGAEA_DOI}"
    "?format=textfile"
)

# PANGAEAの個別ファイル取得に認証が必要な場合だけ使用
PANGAEA_BEARER_TOKEN = os.environ.get(
    "PANGAEA_BEARER_TOKEN",
    "",
)

PANGAEA_HEADERS = {}

if PANGAEA_BEARER_TOKEN:
    PANGAEA_HEADERS["Authorization"] = (
        f"Bearer {PANGAEA_BEARER_TOKEN}"
    )


pangaea_text = (
    read_url_bytes(PANGAEA_TABLE_URL)
    .decode("utf-8", errors="replace")
)

pangaea_lines = pangaea_text.splitlines()

try:
    table_start = next(
        index
        for index, line in enumerate(pangaea_lines)
        if line.strip() == "*/"
    )
except StopIteration as error:
    raise RuntimeError(
        "PANGAEA表の開始位置を取得できません。"
    ) from error


pangaea_table = pd.read_csv(
    StringIO(
        "\n".join(
            pangaea_lines[table_start + 1:]
        )
    ),
    sep="\t",
    dtype=str,
).fillna("")


def extract_nc_reference(row):
    for value in row.tolist():
        text = str(value).strip()

        if ".nc" not in text.lower():
            continue

        if text.lower().startswith(
            ("http://", "https://")
        ):
            parsed = urlparse(text)
            filename = unquote(
                Path(parsed.path).name
            )
            return text, filename

        filename = unquote(
            Path(text).name
        )

        if filename.lower().endswith(".nc"):
            url = (
                "https://download.pangaea.de/dataset/"
                f"{PANGAEA_DATASET_ID}/files/"
                f"{quote(filename)}"
            )
            return url, filename

    return "", ""


def classify_recharge_kind(text):
    normalized = (
        text.lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    if (
        "diffuse groundwater recharge" in normalized
        or "qrdif" in normalized
    ):
        return "diffuse"

    if (
        "groundwater recharge from surface water bodies"
        in normalized
        or "qrswb" in normalized
        or "surface water body" in normalized
    ):
        return "surface_water_body"

    if (
        "total groundwater recharge" in normalized
        or re.search(r"\bqr\b", normalized)
    ):
        return "total"

    return None


candidate_rows = []

for row_index, row in pangaea_table.iterrows():
    url, filename = extract_nc_reference(row)

    if not filename:
        continue

    description = " | ".join(
        str(value)
        for value in row.tolist()
    )

    candidate_rows.append(
        {
            "row_index": row_index,
            "filename": filename,
            "url": url,
            "description": description,
        }
    )

recharge_candidates = pd.DataFrame(candidate_rows)

if recharge_candidates.empty:
    raise RuntimeError(
        "PANGAEA表からNetCDFファイルを見つけられませんでした。"
    )


recharge_candidates["kind"] = recharge_candidates.apply(
    lambda row: classify_recharge_kind(
        row["filename"]
        + " | "
        + row["description"]
    ),
    axis=1,
)

recharge_candidates["is_monthly"] = (
    recharge_candidates["filename"]
    .str.contains(
        r"monthly|month|_mon",
        case=False,
        regex=True,
    )
)

recharge_candidates["is_annual"] = (
    recharge_candidates["filename"]
    .str.contains(
        r"annual|year|_yr|lta",
        case=False,
        regex=True,
    )
)

recharge_candidates.to_csv(
    OUTPUT_DIR
    / "watergap_pangaea_netcdf_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nWaterGAP candidate files")
print(
    recharge_candidates[
        [
            "row_index",
            "filename",
            "kind",
            "is_annual",
            "is_monthly",
        ]
    ].to_string(index=False)
)


# ============================================================
# 5. 涵養量ファイルの選択とダウンロード
# ============================================================

# 自動判定がうまくいかない場合だけ、ここに正確なファイル名を入れる
RECHARGE_FILE_OVERRIDES = {
    # "diffuse": "exact_filename.nc",
    # "surface_water_body": "exact_filename.nc",
    # "total": "exact_filename.nc",
}


def choose_recharge_file(kind):
    candidates = recharge_candidates[
        recharge_candidates["kind"].eq(kind)
    ].copy()

    if candidates.empty:
        if kind == "surface_water_body":
            print(
                "qrswbファイルが見つからないためスキップします。"
            )
            return None

        raise RuntimeError(
            f"{kind}のWaterGAPファイルが見つかりません。"
        )

    override = RECHARGE_FILE_OVERRIDES.get(kind)

    if override:
        exact = candidates[
            candidates["filename"].eq(override)
        ]

        if exact.empty:
            raise RuntimeError(
                f"指定ファイルが候補にありません: {override}"
            )

        return exact.iloc[0]

    # 年ファイルを優先。なければ月ファイルを選択して後で平均。
    candidates["score"] = 0

    candidates.loc[
        candidates["is_annual"],
        "score",
    ] += 100

    candidates.loc[
        candidates["is_monthly"],
        "score",
    ] -= 50

    candidates = candidates.sort_values(
        ["score", "filename"],
        ascending=[False, True],
    )

    selected = candidates.iloc[0]

    print(
        f"{kind} selected:",
        selected["filename"],
    )

    return selected


selected_rows = {}

for kind in [
    "diffuse",
    "surface_water_body",
    "total",
]:
    selected = choose_recharge_file(kind)

    if selected is not None:
        selected_rows[kind] = selected


recharge_raw_paths = {}

for kind, row in selected_rows.items():
    destination = (
        RECHARGE_RAW_DIR
        / row["filename"]
    )

    recharge_raw_paths[kind] = download_file(
        row["url"],
        destination,
        headers=PANGAEA_HEADERS,
        timeout=1800,
    )


selected_manifest = pd.DataFrame(
    [
        {
            "kind": kind,
            "filename": row["filename"],
            "url": row["url"],
            "local_path": str(recharge_raw_paths[kind]),
            "size_bytes": recharge_raw_paths[kind].stat().st_size,
        }
        for kind, row in selected_rows.items()
    ]
)

selected_manifest.to_csv(
    OUTPUT_DIR
    / "watergap_selected_recharge_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nSelected WaterGAP files")
print(selected_manifest.to_string(index=False))


# ============================================================
# 6. WaterGAP涵養量をmm/年に統一して保存
# ============================================================

def find_coordinate(ds, candidates):
    for name in ds.coords:
        if name.lower() in [
            candidate.lower()
            for candidate in candidates
        ]:
            return name

    for name in ds.coords:
        if any(
            candidate.lower() in name.lower()
            for candidate in candidates
        ):
            return name

    return None


def find_recharge_variable(ds, kind):
    exact_aliases = {
        "diffuse": ["qrdif"],
        "surface_water_body": ["qrswb"],
        "total": ["qr"],
    }

    for alias in exact_aliases[kind]:
        for name in ds.data_vars:
            lower_name = name.lower()

            if (
                lower_name == alias
                or lower_name.startswith(alias + "_")
            ):
                return name

    candidates = [
        name
        for name, variable in ds.data_vars.items()
        if variable.ndim >= 2
    ]

    if len(candidates) == 1:
        return candidates[0]

    for name in candidates:
        lower_name = name.lower()

        if kind == "diffuse" and "diffuse" in lower_name:
            return name

        if (
            kind == "surface_water_body"
            and "surface" in lower_name
        ):
            return name

        if kind == "total" and "total" in lower_name:
            return name

    raise RuntimeError(
        f"{kind}のデータ変数を特定できません。"
        f" data_vars={list(ds.data_vars)}"
    )


def select_year_mean(data, start_year, end_year):
    for dim in data.dims:
        if dim.lower() not in [
            "time",
            "date",
            "year",
        ]:
            continue

        coordinate = data[dim]

        # datetime座標
        try:
            years = coordinate.dt.year

            selected = data.where(
                (years >= start_year)
                & (years <= end_year),
                drop=True,
            )

            if selected.sizes.get(dim, 0) > 0:
                return selected.mean(
                    dim=dim,
                    skipna=True,
                )
        except Exception:
            pass

        # 数値型のyear座標
        try:
            numeric_years = np.asarray(
                coordinate.values
            ).astype(int)

            mask = (
                (numeric_years >= start_year)
                & (numeric_years <= end_year)
            )

            if mask.any():
                return data.isel(
                    {dim: np.flatnonzero(mask)}
                ).mean(
                    dim=dim,
                    skipna=True,
                )
        except Exception:
            pass

    return data


def convert_to_mm_per_year(data):
    units = str(
        data.attrs.get("units", "")
    ).lower().replace(" ", "")

    values = data.astype("float32")
    seconds_per_year = 365.25 * 24 * 60 * 60

    conversion = "none"

    if (
        "s-1" in units
        or "/s" in units
        or "s^-1" in units
    ):
        values = values * seconds_per_year
        conversion = "per_second_to_mm_per_year"

    elif (
        "d-1" in units
        or "/d" in units
        or "d^-1" in units
    ):
        values = values * 365.25
        conversion = "per_day_to_mm_per_year"

    elif (
        "month-1" in units
        or "/month" in units
        or "monthly" in units
    ):
        values = values * 12.0
        conversion = "per_month_to_mm_per_year"

    elif (
        "yr-1" in units
        or "/yr" in units
        or "year-1" in units
        or "mm/year" in units
    ):
        conversion = "already_annual"

    else:
        print(
            "単位を判定できないため、値をそのまま保存:",
            units,
        )

    values.attrs["original_units"] = units
    values.attrs["conversion"] = conversion
    values.attrs["units"] = "mm yr-1"

    return values


processed_rows = []

for kind, raw_path in recharge_raw_paths.items():
    with xr.open_dataset(
        raw_path,
        decode_cf=True,
    ) as ds:

        variable_name = find_recharge_variable(
            ds,
            kind,
        )

        data = ds[variable_name]

        data = select_year_mean(
            data,
            RECHARGE_MEAN_START_YEAR,
            RECHARGE_MEAN_END_YEAR,
        )

        data = (
            convert_to_mm_per_year(data)
            .rename(
                f"groundwater_recharge_{kind}_mm_yr"
            )
            .load()
        )

        data.attrs["source"] = (
            "WaterGAP 2.2d Standard model output"
        )

        data.attrs["source_doi"] = PANGAEA_DOI
        data.attrs["source_variable"] = variable_name
        data.attrs["mean_period"] = (
            f"{RECHARGE_MEAN_START_YEAR}-"
            f"{RECHARGE_MEAN_END_YEAR}"
        )

        output_path = (
            RECHARGE_PROCESSED_DIR
            / (
                f"watergap_{kind}_recharge_"
                f"mm_yr_"
                f"{RECHARGE_MEAN_START_YEAR}_"
                f"{RECHARGE_MEAN_END_YEAR}.nc"
            )
        )

        data.to_dataset().to_netcdf(output_path)

        finite = data.values[
            np.isfinite(data.values)
        ]

        processed_rows.append(
            {
                "kind": kind,
                "source_variable": variable_name,
                "path": str(output_path),
                "unit": "mm yr-1",
                "n_finite": finite.size,
                "min": float(np.min(finite)),
                "median": float(np.median(finite)),
                "p90": float(np.percentile(finite, 90)),
                "max": float(np.max(finite)),
            }
        )


recharge_summary = pd.DataFrame(processed_rows)

recharge_summary.to_csv(
    OUTPUT_DIR / "watergap_recharge_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nProcessed recharge summary")
print(recharge_summary.to_string(index=False))


# ============================================================
# 7. 最終マニフェスト
# ============================================================

final_manifest = pd.concat(
    [
        wtd_manifest.assign(
            dataset_group="Fan_WTD"
        ),
        selected_manifest.assign(
            dataset_group="WaterGAP_recharge"
        ),
    ],
    ignore_index=True,
    sort=False,
)

final_manifest.to_csv(
    OUTPUT_DIR / "groundwater_download_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n完了しました。")
print("最終マニフェスト:")
print(OUTPUT_DIR / "groundwater_download_manifest.csv")
print("WTD地域切り出し:")
print(WTD_SUBSET_DIR)
print("涵養量処理済みファイル:")
print(RECHARGE_PROCESSED_DIR)

保存先: /work/tsuda/GAEZ/WaterData/groundwater
WTD mode: diagnostic_regions
カタログ取得失敗: https://thredds-gfnl.usc.es/thredds/catalog/GLOBALWTDFTP/annualmeans/catalog.xml
<urlopen error [Errno 110] Connection timed out>
WTD catalog: http://thredds-gfnl.usc.es/thredds/catalog/GLOBALWTDFTP/annualmeans/catalog.xml
WTD files:
 - GLOBALWTDFTP/annualmeans/AFRICA_WTD_annualmean.nc
 - GLOBALWTDFTP/annualmeans/EURASIA_WTD_annualmean.nc
 - GLOBALWTDFTP/annualmeans/NAMERICA_WTD_annualmean.nc
 - GLOBALWTDFTP/annualmeans/OCEANIA_WTD_annualmean.nc
 - GLOBALWTDFTP/annualmeans/SAMERICA_WTD_annualmean.nc
ダウンロード中: https://thredds-gfnl.usc.es/thredds/fileServer/GLOBALWTDFTP/annualmeans/AFRICA_WTD_annualmean.nc
WTDファイル取得失敗: https://thredds-gfnl.usc.es/thredds/fileServer/
<urlopen error [Errno 110] Connection timed out>
ダウンロード中: http://thredds-gfnl.usc.es/thredds/fileServer/GLOBALWTDFTP/annualmeans/AFRICA_WTD_annualmean.nc
保存: /work/tsuda/GAEZ/WaterData/groundwater/fan_wtd/raw_continents/AFRICA_WTD_annualmean.nc 

KeyboardInterrupt: 